# DPO and GRPO Loss Formulations in PyTorch

This notebook implements DPO (Direct Preference Optimization) implicit reward loss and GRPO (Group Relative Policy Optimization) group advantages normalization.

In [1]:
import torch
import torch.nn.functional as F

# DPO loss calculation
def dpo_loss(policy_win_logps, policy_lose_logps, ref_win_logps, ref_lose_logps, beta=1.0):
    policy_log_ratio = policy_win_logps - policy_lose_logps
    ref_log_ratio = ref_win_logps - ref_lose_logps
    logits = policy_log_ratio - ref_log_ratio
    loss = -F.logsigmoid(beta * logits)
    return loss.mean()

# GRPO relative advantages normalization
def calculate_grpo_advantages(rewards):
    mean_val = rewards.mean()
    std_val = rewards.std() + 1e-8
    advantages = (rewards - mean_val) / std_val
    return advantages

# Test DPO Loss
policy_win = torch.tensor([[-0.5, -0.2]])
policy_lose = torch.tensor([[-1.5, -1.8]])
ref_win = torch.tensor([[-0.8, -0.6]])
ref_lose = torch.tensor([[-1.0, -1.2]])

loss = dpo_loss(policy_win, policy_lose, ref_win, ref_lose, beta=0.1)
print(f"Toy DPO Loss: {loss.item():.4f}")

# Test GRPO Advantages
rewards = torch.tensor([1.0, 2.0, 3.0])
advantages = calculate_grpo_advantages(rewards)
print("\nGRPO Raw Rewards:", rewards.numpy())
print("GRPO Standardized Advantages:", advantages.numpy())


Toy DPO Loss: 0.6492

GRPO Raw Rewards: [1. 2. 3.]
GRPO Standardized Advantages: [-1.  0.  1.]


### Output Explanation
The logs show the calculated DPO loss, and the standardized GRPO advantages. Since output 3 scored the highest ($3.0$), its normalized advantage is positive ($1.22$), reinforcing its generation logits without needing a critic network.